In [191]:
import json
import re

In [357]:
def save_as_json(data, output_file):
    with open(output_file, 'w') as f:
        json.dump(data, f, indent=4)
def save_json_to_file(json_data, file_path):
    with open(file_path, 'w', encoding='utf-8') as f:
        f.write(json_data)

def read_ann_file(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            content = file.read()
            return content
    except FileNotFoundError:
        return "Die Datei konnte nicht gefunden werden."
    except Exception as e:
        return f"Ein Fehler ist aufgetreten: {e}"

In [240]:
def parse_ann_file(file_path):
    entities = {}
    relationships = []
    scope_or = []
    with open(file_path, 'r') as file:
        for line in file:
            parts = line.strip().split('\t')
            if len(parts) < 2:
                continue
            entry_type = parts[0][0]
            if entry_type == 'T':  
                entity_id, entity_info = parts[0], parts[1:]
                label, position = entity_info[0].split(' ')[0], entity_info[0].split(' ')[1:]
                start, end = position[0], position[1]
                if ';' in start:
                    start = start.split(';')[0]
                if ';' in end:
                    end = end.split(';')[0]
                entities[entity_id] = {
                    'type': label,
                    'start': int(start),
                    'end': int(end),
                    'text': parts[2]
                }
            elif entry_type == 'R':  # Relationship
                relation_id, relation_info = parts[0], parts[1]
                relation_type, arg1, arg2 = relation_info.split(' ')[0], relation_info.split(' ')[1].split(':')[1], relation_info.split(' ')[2].split(':')[1]
                relationships.append({
                    'id': relation_id,
                    'type': relation_type,
                    'arg1': arg1,
                    'arg2': arg2
                })
            elif parts[0].startswith('*'):
                relationship_type = parts[1].split()[0]
                entity_ids = parts[1].split()[1:]
                scope_or.append({
                    'type': relationship_type,
                    'entities': entity_ids
                })
    return {
        'entities': entities,
        'relationships': relationships,
        'scope_or': scope_or
    }

In [244]:
def replace_ids_with_offsets(data):
    entities = data['entities']
    relationships = data['relationships']
    scope_or = data['scope_or']
    for relationship in relationships:
        if relationship['type'] in ["AND", "Has_negation"]:
            if relationship['arg1'] in entities and relationship['arg2'] in entities:
                start1, end1 = entities[relationship['arg1']]['start'], entities[relationship['arg1']]['end']
                start2, end2 = entities[relationship['arg2']]['start'], entities[relationship['arg2']]['end']
                # Sortierung der Offsets für konsistente Reihenfolge
                sorted_offsets = sorted([(start1, end1), (start2, end2)])
                relationship['arg1'] = f"{sorted_offsets[0][0]}-{sorted_offsets[0][1]}"
                relationship['arg2'] = f"{sorted_offsets[1][0]}-{sorted_offsets[1][1]}"
    new_scope_or = []
    for group in scope_or:
        offsets = sorted([f"{entities[id]['start']}-{entities[id]['end']}" for id in group['entities'] if id in entities], key=lambda x: int(x.split('-')[0]))
        new_scope_or.append({'type': group['type'], 'entities': offsets})
    updated_data = {
        'entities': entities, 
        'relationships': relationships,
        'scope_or': new_scope_or
    }
    return updated_data

In [245]:
ann_file = 'NCT00050349_exc.ann'
ec_file = 'NCT00050349_exc.txt'

ann_data = parse_ann_file(ann_file)
save_as_json(ann_data, 'output_file.json')
print("Conversion complete. JSON saved to 'output_file.json'")

Conversion complete. JSON saved to 'output_file.json'


In [284]:
def remove_last_elements(data):
    # Entferne das letzte Element aus jeder 'OR' Gruppe, wenn mehr als ein Element vorhanden ist
    for or_group in data['scope_or']:
        if len(or_group['entities']) > 1:
            or_group['entities'].pop()  # Entfernt das letzte Element

    # Entferne das letzte Argument aus den 'AND' und 'Has_negation' Beziehungen
    for relationship in data['relationships']:
        if relationship['type'] in ["AND", "Has_negation"]:
            arg1_end = int(relationship['arg1'].split('-')[1])
            arg2_end = int(relationship['arg2'].split('-')[1])
            if arg1_end > arg2_end:
                relationship['arg1'] = relationship['arg2']  # Setze arg1 auf arg2, wenn arg1 das letzte ist
            # Entferne arg2, da wir nur das erste Argument behalten
            del relationship['arg2']

    return data

In [285]:
offsets = replace_ids_with_offsets(ann_data)
offsets = remove_last_elements(offsets)

KeyError: 'arg2'

In [286]:
offsets

{'entities': {'T1': {'type': 'Condition',
   'start': 26,
   'end': 40,
   'text': 'CNS metastases'},
  'T2': {'type': 'Condition',
   'start': 44,
   'end': 70,
   'text': 'leptomeningeal involvement'},
  'T5': {'type': 'Procedure', 'start': 144, 'end': 151, 'text': 'treated'},
  'T6': {'type': 'Qualifier',
   'start': 164,
   'end': 179,
   'text': 'been stable for'},
  'T7': {'type': 'Temporal',
   'start': 180,
   'end': 220,
   'text': 'at least six months prior to study start'},
  'T4': {'type': 'Condition',
   'start': 92,
   'end': 108,
   'text': 'brain metastases'},
  'T8': {'type': 'Observation',
   'start': 238,
   'end': 248,
   'text': 'history of'},
  'T9': {'type': 'Condition',
   'start': 249,
   'end': 265,
   'text': 'brain metastases'},
  'T10': {'type': 'Procedure',
   'start': 278,
   'end': 299,
   'text': 'head CT with contrast'},
  'T11': {'type': 'Scope',
   'start': 238,
   'end': 265,
   'text': 'history of brain metastases'},
  'T12': {'type': 'Condition',


In [287]:
def insert_character_at_offsets(file_path, data):
    all_offsets = []
    label_for_offsets = {}
    inserted_positions = set()  # Zum Speichern bereits eingefügter Positionen
    # Verarbeiten von AND und Has_negation Beziehungen
    for relationship in data['relationships']:
        if relationship['type'] in ["AND", "Has_negation"]:
            arg1_offset = f"{relationship['arg1']}"
            all_offsets.append(arg1_offset)
            label_for_offsets[arg1_offset] = ' [AND]' if relationship['type'] == "AND" else ' [NEG]'
    # Verarbeiten von OR Gruppen
    for or_group in data['scope_or']:
        if len(or_group['entities']) == 1:
            # Nur ein Offset in der OR Gruppe, füge [OR] danach ein
            single_offset = or_group['entities'][0]
            label_for_offsets[single_offset] = ' [OR]'
            all_offsets.append(single_offset)
        else:
            # Mehrere Offsets, füge [OR] zwischen benachbarten Offsets ein
            for i in range(len(or_group['entities']) - 1):  # Letztes Element überspringen
                current_offset = or_group['entities'][i]
                next_offset = or_group['entities'][i + 1]
                # Füge [OR] nach dem aktuellen Offset ein
                label_for_offsets[current_offset] = ' [OR]'
                all_offsets.append(current_offset)
                # Füge [OR] vor dem nächsten Offset ein, falls nicht schon vorhanden
                if next_offset not in label_for_offsets:
                    label_for_offsets[next_offset] = ' [OR]'
                    all_offsets.append(next_offset)
    # Datei lesen
    with open(file_path, 'r') as file:
        content = file.read()
    # Sortieren der Offsets in umgekehrter Reihenfolge, um die Positionen korrekt zu aktualisieren
    sorted_offsets = sorted([(int(offset.split('-')[1]), offset) for offset in all_offsets], reverse=True)
    # Einfügen der Labels in den Text
    for end_pos, offset in sorted_offsets:
        if end_pos not in inserted_positions:  # Überprüfe, ob das Tag bereits eingefügt wurde
            label = label_for_offsets[offset]
            content = content[:end_pos] + label + content[end_pos:]
            inserted_positions.add(end_pos)
    # Ausgabe in eine neue Datei schreiben
    with open("output_with_tags.txt", 'w') as file:
        file.write(content)
    return content

In [288]:
content = insert_character_at_offsets(ec_file, offsets)

In [289]:
content

'Patients with symptomatic CNS metastases [OR] or leptomeningeal involvement \nPatients with known brain metastases [AND], unless [NEG] these metastases have been treated [OR] and/or have been stable for at least six months prior to study start. Subjects with a history of [AND] brain metastases [AND] must have a head CT with contrast to document either response or progression. \nPatients with bone metastases [AND] as the only site(s) of measurable disease \nPatients with hepatic artery chemoembolization within the last 6 months [OR] (one month if there are other sites of measurable disease) \nPatients who have been previously treated with radioactive directed therapies \nPatients who have been previously treated with epothilone \nPatients with any peripheral neuropathy [OR] or unresolved diarrhea [AND] greater than Grade 1 \nPatients with severe cardiac insufficiency [AND] patients taking Coumadin [OR] or other warfarin-containing agents [AND] with the exception of [NEG] low dose [OR] 

In [331]:
def ec_to_json(text):
    sentences = text.strip().split('\n')
    sentences = [sentence.strip() for sentence in sentences if sentence.strip()]
    data = {f"EC{i+1}": sentence for i, sentence in enumerate(sentences)}
    json_output = json.dumps(data, indent=2, ensure_ascii=False)
    with open("output.json", "w", encoding="utf-8") as file:
        file.write(json_output)
    return json_output

In [333]:
data = ec_to_json(content)

In [370]:
def parse_to_structured_json(input_json):
    data = json.loads(input_json)

    def process_section(text, path):
        segments = re.split(r'(\[OR\]|\[AND\]|\[NEG\])', text)
        structure = {}
        operator = None
        elements = []

        # Durchlaufen der Segmente und Erkennen von Operatoren
        for segment in segments:
            segment = segment.strip()
            if segment in ['[OR]', '[AND]', '[NEG]']:
                operator = "OR" if segment == '[OR]' else "AND" if segment == '[AND]' else "NOT"
            else:
                clean_segment = re.sub(r'\[\w+\]', '', segment).strip()
                if clean_segment:
                    elements.append(clean_segment)

        # Zuordnung der Elemente zur Struktur unter Beachtung des Operators
        if operator:
            structure['operator'] = operator
            for index, element in enumerate(elements, 1):
                structure[f"{path}.{index}"] = element
        else:
            # Wenn kein Operator vorhanden ist, füge alle Elemente direkt hinzu, ohne weitere Unterteilung
            if elements:
                structure = ' '.join(elements)

        return structure

    result_structure = {"EC": {}}
    index = 1

    # Iterieren über die Eingabe und Verarbeiten jedes Abschnitts
    for key, value in data.items():
        section_path = f"EC{index}"
        result_structure["EC"][section_path] = process_section(value, section_path)
        index += 1

    return json.dumps(result_structure, indent=4, ensure_ascii=False)

In [371]:
parsed_data = parse_to_structured_json(data)
print(parsed_data)
save_json_to_file(parsed_data, 'parsed_data.json')

{
    "EC": {
        "EC1": {
            "operator": "OR",
            "EC1.1": "Patients with symptomatic CNS metastases",
            "EC1.2": "or leptomeningeal involvement"
        },
        "EC2": {
            "operator": "AND",
            "EC2.1": "Patients with known brain metastases",
            "EC2.2": ", unless",
            "EC2.3": "these metastases have been treated",
            "EC2.4": "and/or have been stable for at least six months prior to study start. Subjects with a history of",
            "EC2.5": "brain metastases",
            "EC2.6": "must have a head CT with contrast to document either response or progression."
        },
        "EC3": {
            "operator": "AND",
            "EC3.1": "Patients with bone metastases",
            "EC3.2": "as the only site(s) of measurable disease"
        },
        "EC4": {
            "operator": "OR",
            "EC4.1": "Patients with hepatic artery chemoembolization within the last 6 months",
            "E